<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-12c-tensorrt-llm-benchmarking-and-serving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 12 (continued) - TensorRT-LLM: the production GPU stack, and how to benchmark it honestly

Decode-speed ceilings per GPU and precision, a static-versus-in-flight batching simulator, a benchmark harness for any OpenAI-compatible server (vLLM or `trtllm-serve`), and a cost-per-million-tokens calculator.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything in Parts 1-4 runs on a CPU, including the free Colab tier. **Part 5 needs an NVIDIA GPU of the Ampere generation or newer** (A10, A100, L4, H100 ...). The free-tier T4 is a Turing GPU and is *not* in TensorRT-LLM's support matrix, so Part 5 is skipped automatically there.

> ## ⚠️ Part 5 needs a paid Colab plan
> Parts 1-4 run free (CPU). **Part 5 runs TensorRT-LLM and vLLM on a real GPU and needs a Colab Pro / Pro+ (or pay-as-you-go) account** with an Ampere-or-newer runtime (L4 or A100 - choose it under *Runtime > Change runtime type*). The free-tier T4 is not supported by TensorRT-LLM. Paid GPU availability and compute-unit cost are set by Google and change; check your balance first and **disconnect the runtime when you finish**. If you prefer not to pay, Parts 1-4 teach the same method, and the free-credit cloud route from Chapter 15 is an alternative.

*Original code, written for this course. Product facts are taken from NVIDIA's public TensorRT-LLM documentation (Apache-2.0 project); links are in the chapter.*

In [ ]:
%pip install -q numpy requests

In [ ]:
import numpy as np, time, json, threading, statistics
from http.server import BaseHTTPRequestHandler, HTTPServer
np.random.seed(0)

## Part 1 - The decode ceiling by GPU and precision

Chapter 12 showed that batch-1 decode speed is bounded by `memory bandwidth / bytes of weights`. A production stack changes the *bytes* (lower precision) and keeps the *batch* full. Here is the ceiling for an 8-billion-parameter model.

The bandwidth figures are public datasheet numbers (HBM/GDDR peak). Real systems reach a fraction of them.

In [ ]:
GPUS = {                      # peak memory bandwidth, GB/s  (public datasheet values)
    "L4 (24 GB)":        300,
    "A10 (24 GB)":       600,
    "A100 80GB (SXM)":  2039,
    "H100 80GB (SXM)":  3350,
}
BYTES_PER_PARAM = {"FP16/BF16": 2.0, "FP8": 1.0, "INT4 (AWQ, g128)": 0.5 + 2/128}   # +fp16 scale per 128 weights

def ceiling_tok_s(params_b, bw_gbs, bytes_per_param):
    return bw_gbs * 1e9 / (params_b * 1e9 * bytes_per_param)

print(f"{'GPU':18}" + "".join(f"{p:>18}" for p in BYTES_PER_PARAM))
for g, bw in GPUS.items():
    print(f"{g:18}" + "".join(f"{ceiling_tok_s(8, bw, b):>18.0f}" for b in BYTES_PER_PARAM.values()))
print("\n(8B parameters, batch 1, tokens/s upper bound; weights only)")

**Read it.** Moving 16-bit weights to FP8 doubles the ceiling; INT4 with group scales lifts it by almost 4x. But FP8 needs a GPU that has FP8 hardware (Ada or Hopper and newer), while weight-only INT4 AWQ/GPTQ runs on Ampere as well. *Which formats a stack offers depends on the GPU generation* - that is the first thing to check before choosing a quantization recipe.

## Part 2 - Static batching versus in-flight (continuous) batching

TensorRT-LLM, vLLM and others all use in-flight batching: when one sequence finishes, a waiting request takes its slot at the next step instead of waiting for the whole batch. We simulate it with a simple cost model: one decode step takes `t_w` (streaming the weights, shared by every sequence in the step) plus a small `t_kv` per active sequence (reading its KV cache).

In [ ]:
def simulate(out_lens, slots, t_w, t_kv, mode):
    """Return (total_time_s, tokens_generated, per-request latencies). All requests arrive at t=0."""
    pending = list(range(len(out_lens))); active = {}; t = 0.0; done_at = {}
    while pending or active:
        if mode == "static":
            if not active:                                  # start a new batch only when the old one is fully done
                for _ in range(min(slots, len(pending))):
                    i = pending.pop(0); active[i] = out_lens[i]
        else:                                               # in-flight: refill free slots every step
            while pending and len(active) < slots:
                i = pending.pop(0); active[i] = out_lens[i]
        n = len(active)
        t += t_w + n * t_kv
        for i in list(active):
            active[i] -= 1
            if active[i] == 0:
                del active[i]; done_at[i] = t
    return t, sum(out_lens), [done_at[i] for i in range(len(out_lens))]

rng = np.random.default_rng(1)
out_lens = np.clip(rng.lognormal(mean=4.6, sigma=0.8, size=256).astype(int), 8, 800).tolist()  # long-tailed output lengths
print("requests:", len(out_lens), "| mean output len:", round(np.mean(out_lens)), "| max:", max(out_lens))

t_w = 16e9 / 2039e9          # 8B FP16 weights on an A100: ~7.8 ms per step
t_kv = 0.00005               # 50 microseconds of KV reading per active sequence per step (illustrative)
for slots in (8, 32, 64):
    ts, n, ls = simulate(out_lens, slots, t_w, t_kv, "static")
    ti, _, li = simulate(out_lens, slots, t_w, t_kv, "inflight")
    print(f"slots={slots:3d}  static: {n/ts:7.0f} tok/s, mean latency {np.mean(ls):6.1f}s   |   in-flight: {n/ti:7.0f} tok/s, mean latency {np.mean(li):6.1f}s   (x{(n/ti)/(n/ts):.2f} throughput)")

The static scheduler above keeps a batch running until its longest request ends; the sequences that finished early leave their slot idle, so fewer sequences share each pass over the weights. (This simulator charges only for active sequences and so *understates* the real cost of static batching, where finished rows are padded and still computed. The throughput gap you see is therefore a lower bound.) Because output lengths are long-tailed, in-flight batching wins by a wide margin,  the in-flight scheduler delivered about 2.3x to 2.8x the throughput of the static one at every slot count we tried (your exact ratio depends on the length distribution).

## Part 3 - A benchmark harness for any OpenAI-compatible server

Both vLLM (`vllm serve`) and TensorRT-LLM (`trtllm-serve`) expose `/v1/chat/completions`. A harness that only speaks that protocol lets you compare engines *without changing the harness* - the fair-comparison rule. We measure what users feel and what the invoice shows:

* **TTFT** - time to first token (queueing + prefill).
* **ITL** - inter-token latency during decode.
* **Output tokens/s** per request, and **aggregate** output tokens/s across all clients.
* Percentiles (p50, p95, p99), because averages hide the slow tail.

To keep this runnable anywhere, we first point it at a tiny *mock* server whose speed follows the cost model from Part 2. Part 5 shows how to point the very same function at a real server.

In [ ]:
class Mock(BaseHTTPRequestHandler):
    t_w, t_kv, lock, active = 0.004, 0.0002, threading.Lock(), 0
    def log_message(self, *a): pass
    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        n_out = body.get("max_tokens", 32)
        self.send_response(200); self.send_header("Content-Type", "text/event-stream"); self.end_headers()
        with Mock.lock: Mock.active += 1
        try:
            time.sleep(0.01 + 0.0001 * len(json.dumps(body["messages"])))          # "prefill"
            for k in range(n_out):
                with Mock.lock: a = Mock.active
                time.sleep(Mock.t_w + a * Mock.t_kv)                                # one decode step, slower when busier
                self.wfile.write(b"data: " + json.dumps({"choices":[{"delta":{"content":"tok "}}]}).encode() + b"\n\n"); self.wfile.flush()
            self.wfile.write(b"data: [DONE]\n\n"); self.wfile.flush()
        finally:
            with Mock.lock: Mock.active -= 1

class Threaded(HTTPServer):
    def process_request(self, req, addr):
        threading.Thread(target=self.process_request_thread, args=(req, addr), daemon=True).start()
    def process_request_thread(self, req, addr):
        try: self.finish_request(req, addr)
        finally: self.shutdown_request(req)

srv = Threaded(("127.0.0.1", 8765), Mock); threading.Thread(target=srv.serve_forever, daemon=True).start()

In [ ]:
import requests

def one_request(base_url, model, prompt, max_tokens, out):
    t0 = time.perf_counter(); stamps = []
    with requests.post(base_url + "/v1/chat/completions", stream=True, timeout=300,
                       json={"model": model, "messages": [{"role": "user", "content": prompt}],
                             "max_tokens": max_tokens, "stream": True}) as r:
        for line in r.iter_lines(chunk_size=1):
            if line.startswith(b"data: ") and line != b"data: [DONE]":
                stamps.append(time.perf_counter())
    if stamps:
        out.append({"ttft": stamps[0] - t0, "itl": np.diff(stamps).tolist(), "n": len(stamps), "end": stamps[-1] - t0})

def benchmark(base_url, model, concurrency, n_requests, prompt, max_tokens, warmup=4):
    for _ in range(warmup):                                  # warm-up requests are discarded
        one_request(base_url, model, prompt, 4, [])
    results, sem = [], threading.Semaphore(concurrency)
    def worker():
        with sem: one_request(base_url, model, prompt, max_tokens, results)
    t0 = time.perf_counter()
    ts = [threading.Thread(target=worker) for _ in range(n_requests)]
    [t.start() for t in ts]; [t.join() for t in ts]
    wall = time.perf_counter() - t0
    itl = np.concatenate([r["itl"] for r in results]) * 1000
    ttft = np.array([r["ttft"] for r in results]) * 1000
    return {"concurrency": concurrency, "agg_tok_s": sum(r["n"] for r in results) / wall,
            "ttft_p50_ms": np.percentile(ttft, 50), "ttft_p95_ms": np.percentile(ttft, 95),
            "itl_p50_ms": np.percentile(itl, 50), "itl_p99_ms": np.percentile(itl, 99)}

print(f"{'conc':>5} {'agg tok/s':>10} {'TTFT p50':>9} {'TTFT p95':>9} {'ITL p50':>8} {'ITL p99':>8}")
for c in (1, 8, 32):
    r = benchmark("http://127.0.0.1:8765", "mock", c, max(2 * c, 8), "Summarise the weekly maintenance log for line 3.", 48)
    print(f"{r['concurrency']:>5} {r['agg_tok_s']:>10.0f} {r['ttft_p50_ms']:>8.0f}ms {r['ttft_p95_ms']:>8.0f}ms {r['itl_p50_ms']:>7.1f}ms {r['itl_p99_ms']:>7.1f}ms")

**The shape to expect from any real server:** aggregate throughput climbs with concurrency while per-user inter-token latency worsens. The "right" operating point is set by your latency promise (for example ITL p99 under 50 ms), *not* by the highest throughput number. Always report the concurrency, input/output lengths, precision, GPU and engine version next to every figure.

## Part 4 - Cost per million output tokens

Throughput only matters once it is turned into money. `cost = GPU price per hour / (tokens per hour)`. Fill in *your* measured aggregate tokens/s at your latency target.

In [ ]:
def usd_per_million_tokens(gpu_usd_per_hour, agg_tok_s, utilisation=0.6):
    """utilisation = fraction of the hour the GPU is actually serving traffic at that rate."""
    return gpu_usd_per_hour / (agg_tok_s * 3600 * utilisation) * 1e6

# Illustrative inputs - replace with your cloud price and your own measured throughput.
for label, price, tok_s in [("engine A, FP16", 2.0, 1800), ("engine B, FP8 (same GPU)", 2.0, 3200)]:
    print(f"{label:28}  ${usd_per_million_tokens(price, tok_s):6.2f} per 1M output tokens at 60% utilisation")

A 1.8x throughput gain at the same latency target is a 44% cut in cost per token. Leaving the GPU idle half the day costs more than most engine choices save - utilisation belongs in the same sentence as throughput.

## Part 5 - OPTIONAL, PAID-GPU: TensorRT-LLM versus vLLM on a real GPU

**Requires Colab Pro / Pro+ (or pay-as-you-go compute units) with an L4 or A100 runtime.** Skipped automatically if no Ampere-or-newer GPU is visible.

**Honest caveats.** TensorRT-LLM pins specific CUDA and PyTorch versions and changes quickly; installing it with `pip` inside Colab may fail if Colab's preinstalled versions differ. If it does, use NVIDIA's NGC container on a cloud GPU (see the chapter), or run the vLLM half only. We could not pre-test every Colab image, so treat the install cell as a starting point. The model is a ~1B-parameter chat model so that both engines fit on an L4 and the run costs only a few minutes of compute.

In [ ]:
import subprocess, shutil, os, sys
cc = None
if shutil.which("nvidia-smi"):
    try:
        cc = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True).strip().splitlines()[0]
    except Exception:
        cc = None
print("GPU:", cc or "none detected")
supported = bool(cc) and float(cc.split(",")[-1]) >= 8.0
print("Ampere or newer (compute capability >= 8.0):", supported)
if not supported:
    print("Skipping Part 5 - Parts 1-4 are the free, offline path. (On Colab: Runtime > Change runtime type > L4 or A100; needs a paid plan.)")

In [ ]:
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"      # small open model; swap for a larger one if your GPU allows
if supported:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorrt_llm", "vllm"], check=False)   # may need version pins - see the caveats above

In [ ]:
def serve_and_bench(cmd, port, label, levels=(1, 8, 32), max_tokens=128, boot_s=600):
    proc = subprocess.Popen(cmd, stdout=open(f"/tmp/{label}.log", "w"), stderr=subprocess.STDOUT)
    url = f"http://127.0.0.1:{port}"
    try:
        t0 = time.time()
        while time.time() - t0 < boot_s:
            try:
                if requests.get(url + "/health", timeout=2).status_code == 200: break
            except Exception: pass
            if proc.poll() is not None: raise RuntimeError(f"{label} exited early - see /tmp/{label}.log")
            time.sleep(5)
        else:
            raise TimeoutError(f"{label} did not become healthy in {boot_s}s")
        rows = [benchmark(url, MODEL, c, max(2 * c, 8), "Summarise the weekly maintenance log for production line 3 in two sentences.", max_tokens) for c in levels]
        return rows
    finally:
        proc.terminate(); proc.wait(timeout=60)      # free the GPU for the next engine

results = {}
if supported:
    try:
        results["TensorRT-LLM"] = serve_and_bench(["trtllm-serve", MODEL, "--backend", "pytorch", "--max_batch_size", "64", "--max_num_tokens", "4096", "--port", "8000"], 8000, "trtllm")
    except Exception as e:
        print("TensorRT-LLM run failed:", e)
    try:
        results["vLLM"] = serve_and_bench(["vllm", "serve", MODEL, "--port", "8001", "--max-num-seqs", "64"], 8001, "vllm")
    except Exception as e:
        print("vLLM run failed:", e)

In [ ]:
if results:
    print(f"GPU: {cc} | model: {MODEL} | ISL ~20 tokens | OSL 128 tokens")
    print(f"{'engine':14}{'conc':>5}{'agg tok/s':>11}{'TTFT p50':>10}{'TTFT p95':>10}{'ITL p50':>9}{'ITL p99':>9}")
    for eng, rows in results.items():
        for r in rows:
            print(f"{eng:14}{r['concurrency']:>5}{r['agg_tok_s']:>11.0f}{r['ttft_p50_ms']:>8.0f}ms{r['ttft_p95_ms']:>8.0f}ms{r['itl_p50_ms']:>7.1f}ms{r['itl_p99_ms']:>7.1f}ms")
else:
    print("No live results (no supported GPU, or both servers failed to start). Parts 1-4 above still stand.")

## What to hand in if you did this on real hardware

A table with **GPU, driver, engine and version, precision, input/output lengths, concurrency, aggregate tokens/s, TTFT p50/p95, ITL p50/p99**, plus the command lines that produced it (the printed table above plus the output of `pip list` filtered for tensorrt, vllm and torch). One sentence on what you would change first to cut cost per token. A comparison without these columns cannot be reproduced and should not be trusted - including ours.

**Remember to disconnect the runtime (Runtime > Disconnect and delete runtime) so paid compute units stop running.**